In [ ]:
# %pip install fitz
# %pip install PyMuPDF
import pymupdf as fitz
from pathlib import Path

In [ ]:
resume_path = Path("../data/sample_resumes/sample_resume.pdf")
doc = fitz.open(resume_path)
text = ""
for page in doc:
    text += page.get_text()

print(text)

In [ ]:
import re
def clean_text(text):
    # normalize line endings
    text = text.replace("\r\n", "\n")
    text = text.replace("\r", "\n")

    # remove excessive spaces
    text = re.sub(r"[ \t]+", " ", text)

    # remove spaces at the beginning/end of lines
    text = "\n".join(line.strip() for line in text.split("\n"))

    # remove excessive blank lines
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()

In [ ]:
cleaned_text = clean_text(text)

print(cleaned_text)

In [ ]:
print("Original characters:", len(text))
print("Cleaned characters:", len(cleaned_text))
print("Original lines:", len(text.splitlines()))
print("Cleaned lines:", len(cleaned_text.splitlines()))

In [ ]:
SECTION_NAMES = {
    "education": [
        "education",
        "academic background",
        "academics"
    ],

    "experience": [
        "experience",
        "work experience",
        "professional experience",
        "internship",
        "internships"
    ],

    "projects": [
        "projects",
        "academic projects",
        "personal projects",
        "project experience"
    ],

    "skills": [
        "skills",
        "technical skills",
        "core skills"
    ],

    "certifications": [
        "certifications",
        "certificates"
    ],

    "achievements": [
        "achievements",
        "accomplishments",
        "awards",
    ]
}

In [ ]:
def detect_sections(text):
    sections = {}

    lines = text.splitlines()

    for i, line in enumerate(lines):
        normalized = line.lower().strip()

        for section, possible_names in SECTION_NAMES.items():
            if normalized in possible_names:
                sections[section] = i

    return sections

In [ ]:
section_positions = detect_sections(cleaned_text)

section_positions

In [ ]:
def extract_sections(text, section_positions):
    lines = text.splitlines()

    sorted_sections = sorted(
        section_positions.items(),
        key=lambda x: x[1]
    )

    extracted = {}

    for i, (section_name, start_index) in enumerate(sorted_sections):

        if i + 1 < len(sorted_sections):
            end_index = sorted_sections[i + 1][1]
        else:
            end_index = len(lines)

        content = lines[start_index + 1:end_index]

        extracted[section_name] = "\n".join(content).strip()

    return extracted

In [ ]:
sections = extract_sections(
    cleaned_text,
    section_positions
)

In [ ]:
for section, content in sections.items():
    print("=" * 60)
    print(section.upper())
    print("=" * 60)
    print(content)
    print()

In [ ]:
resume_data = {
    "raw_text": cleaned_text,
    "sections": sections
}

In [ ]:
resume_data.keys()

In [ ]:
resume_data["sections"].keys()

In [ ]:
# %pip install pydantic
from pydantic import BaseModel, Field
from typing import List, Optional

In [ ]:
## Candidate Profile Schema

class Education(BaseModel):
    degree: str
    institution: str
    cgpa: Optional[float] = None
    graduation_year: Optional[int] = None


class Project(BaseModel):
    name: str
    description: str
    technologies: List[str] = Field(default_factory=list)


class Experience(BaseModel):
    role: str
    organization: str
    description: str
    technologies: List[str] = Field(default_factory=list)


class CandidateProfile(BaseModel):
    name: str
    email: Optional[str] = None
    phone: Optional[str] = None

    education: List[Education] = Field(default_factory=list)
    skills: List[str] = Field(default_factory=list)
    projects: List[Project] = Field(default_factory=list)
    experience: List[Experience] = Field(default_factory=list)
    certifications: List[str] = Field(default_factory=list)
    achievements: List[str] = Field(default_factory=list)

In [ ]:
profile = CandidateProfile(
    name="Test Candidate",
    skills=["Python", "SQL"],
    education=[
        Education(
            degree="B.Tech CSE",
            institution="Test University",
            cgpa=8.2
        )
    ]
)

profile

In [ ]:
profile.model_dump()

In [ ]:
profile = CandidateProfile(
    name="Test Candidate",
    skills=["Python"],
    projects=[
        Project(
            name="Test Project",
            description="A test project",
            technologies=["Python", "FastAPI"]
        )
    ]
)

print(profile.model_dump())

In [ ]:
# profile = CandidateProfile(
#     name="Test Candidate",
#     education=[
#         Education(
#             degree="B.Tech",
#             institution="University",
#             cgpa="not a number"
#         )
#     ]
# )            # test for validation error

In [ ]:
def build_resume_context(resume_data):
    context = ""

    for section_name, content in resume_data["sections"].items():
        context += f"\n### {section_name.upper()}\n"
        context += content
        context += "\n"

    return context

In [ ]:
resume_context = build_resume_context(resume_data)

print(resume_context)

In [ ]:
# schema test
profile = CandidateProfile(
    name="Test Candidate",
    skills=["Python", "SQL"]
)

print(profile.model_dump())

In [ ]:
# pip install groq
# %pip install dotenv

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv("../.env")

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

print("API key loaded:", GROQ_API_KEY is not None)

In [ ]:
from groq import Groq

client = Groq(api_key=GROQ_API_KEY)

In [ ]:
RESUME_EXTRACTION_PROMPT = """
You are a resume information extraction system.

Your task is to extract ALL factual information explicitly present in the provided resume and return it as a structured JSON object that exactly matches the CandidateProfile schema.

IMPORTANT:
- The CandidateProfile schema is the ONLY source of truth for the JSON structure.
- Use EXACTLY the field names defined in the schema.
- Never rename a field.
- Never create additional fields that are not present in the schema.
- Never use alternative field names.
- Do not return markdown, explanations, comments, or ```json fences.
- Return ONLY the JSON object.

GENERAL RULES:
1. Extract only information explicitly present in the resume.
2. Never invent, infer, assume, or hallucinate information.
3. Preserve the candidate's wording where appropriate.
4. Extract as much information as possible from the resume.
5. If a string field has no corresponding information, use "".
6. If a list field has no corresponding information, use [].
7. If a nullable field has no corresponding information, use null.
8. Do not omit required fields.
9. Do not add fields that are not defined in CandidateProfile.
10. Keep dates, names, organizations, technologies, links, and descriptions faithful to the resume.

SKILLS:
- Extract all explicitly listed technical and professional skills.
- Preserve individual skills as separate items when appropriate.
- Do not infer additional skills from projects or experience.
- Do not add a technology merely because it would normally be associated with a project.

EDUCATION:
- Extract every education entry.
- Extract the exact degree/program name.
- Extract the institution name.
- Extract CGPA only when explicitly mentioned.
- Extract graduation year only when explicitly mentioned.
- If CGPA or graduation year is absent, use null.

EXPERIENCE:
- Extract internships, jobs, work experience, research experience, and relevant organizational/professional experience.
- Keep each experience as a separate entry.
- Extract the exact role/title.
- Extract the organization/company name.
- Preserve the description of responsibilities and work.
- Extract technologies explicitly mentioned in that experience.
- Do not infer technologies.

PROJECTS:
- Extract EVERY project mentioned in the resume.
- Each project must use the field `name`, NOT `title`.
- Extract the project name exactly or as closely as possible to the resume.
- Extract the project's description.
- Extract technologies explicitly associated with that project.
- Do not infer technologies from the general skills section.
- If the resume contains project links but the CandidateProfile schema does not contain a field for them, do not create a new field for those links.
- Do not use fields such as `title`, `github`, `liveDemo`, `link`, or `url` unless those fields explicitly exist in CandidateProfile.

CERTIFICATIONS:
- Extract every explicitly mentioned certification.
- Preserve certification names accurately.

ACHIEVEMENTS:
- Extract every explicitly mentioned achievement, award, competition result, publication, recognition, or similar accomplishment that belongs in the achievements section.
- Do not convert ordinary responsibilities into achievements.

POSITIONS OF RESPONSIBILITY:
- Extract leadership roles, club positions, committee positions, student organization roles, and other explicitly stated positions of responsibility.
- Keep them separate from professional experience unless the schema indicates otherwise.

CONTACT INFORMATION:
- Extract the candidate's name exactly as written.
- Extract email, phone number, address, LinkedIn, GitHub, or other contact information ONLY if corresponding fields exist in CandidateProfile.
- Do not fabricate missing contact information.

SUMMARY:
- Extract the candidate's existing professional/profile summary if present.
- Do not write a new summary.
- Do not infer a summary from the resume if one is not explicitly provided.

CRITICAL PROJECT FIELD RULE:
If the CandidateProfile schema defines:
    "name"
then the JSON MUST contain:
    "name"

NEVER output:
    "title"

Similarly, use the exact field names from CandidateProfile for every other object and field.

The final response MUST be valid JSON and MUST conform to the CandidateProfile schema.

Resume:
"""

In [ ]:
import json

schema = CandidateProfile.model_json_schema()

prompt = RESUME_EXTRACTION_PROMPT + f"""

CandidateProfile JSON Schema:
{json.dumps(schema, indent=2)}

Now extract the information from the resume and return ONLY the JSON object.

Resume:
{cleaned_text}
"""

In [ ]:
response = client.chat.completions.create(
    model="openai/gpt-oss-120b",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ],
    temperature=0
)

In [ ]:
response_text = response.choices[0].message.content

print(response_text)

In [ ]:
parsed_data = json.loads(response_text)

print(parsed_data)

In [ ]:
candidate_profile = CandidateProfile.model_validate(parsed_data)

In [ ]:
candidate_profile

In [ ]:
candidate_data=candidate_profile.model_dump()

In [ ]:
import json

with open(
    "../data/candidate_profile.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        candidate_data,
        f,
        indent=2,
        ensure_ascii=False
    )